# ERA LTE × climate-toolkit — Colab walkthrough

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CGIAR-Climate-Data-Hub/climate-toolkit/blob/main/examples/era_lte_colab.ipynb)

Validate the toolkit's climate metrics against ERA's long-term-experiment (LTE) field records, then explore how yield relates to climate — with plots.

Credential-free by default (NASA POWER + the public ERA `lte_final.csv`). The section-4 trend example is set to the Earth Engine source `era_5` — do section 2 first, or switch its `--source` to `nasa_power` to run it credential-free. Then run the cells top to bottom.

In [ ]:
%pip install -q climate-toolkit

## 1. Install

Install the `climate-toolkit` package from PyPI (a few seconds on Colab). The ERA workflow scripts come from a quick repo clone in the next section.

In [ ]:
import climate_toolkit as ct

print(f"climate_toolkit v{ct.__version__}")
print("Public API:", [n for n in ct.__all__ if not n.startswith("__")])

## 2. Set up Google Earth Engine access (foundational)

Most gridded and projection sources (`agera_5`, `era_5`, `chirps_*`, `imerg`, `terraclimate`, `cmip_6`, `nex_gddp`, ...) route through **Google Earth Engine**. This is the one piece of setup the toolkit needs — do it once and it works everywhere (Colab, your laptop, servers). It is **free for noncommercial use** (research, academia, nonprofit) and needs no credit card:

1. Go to https://code.earthengine.google.com/register with your Google account.
2. Choose **"Register a Noncommercial or Commercial Cloud project"**, then **create a new Google Cloud project** (or reuse one).
3. Usage type: **Unpaid usage** (do *not* pick "Paid usage"). Category: e.g. **Academia & Research** or **Nonprofit**.
4. Note the **project id** you registered (looks like `my-project-123456`), paste it below, set `RUN_EARTH_ENGINE = True`, and run the cell. Colab pops up a Google sign-in.

Full walkthrough with screenshots: [Getting started → Google Earth Engine credentials](https://CGIAR-Climate-Data-Hub.github.io/climate-toolkit/getting_started/#2-google-earth-engine-credentials).

> **No account yet?** Leave `RUN_EARTH_ENGINE = False` and keep going — sections 3 and 5 run credential-free (NASA POWER). The section-4 example is set to `era_5` (Earth Engine); switch its `--source` to `nasa_power` to run it without this setup. Section 6 cells are skipped until you complete this step.

In [ ]:
RUN_EARTH_ENGINE = False  # set True once you have registered (free for noncommercial use)
GCP_PROJECT_ID = "your-ee-project-id"  # <-- your registered Cloud project id

# Tip: instead of pasting the id here, store it once in Colab's Secrets
# (key icon in the left sidebar, name it GCP_PROJECT_ID) and use:
#   from google.colab import userdata
#   GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")

if RUN_EARTH_ENGINE:
    import os

    import ee

    ee.Authenticate()  # interactive Google sign-in (works natively in Colab)
    os.environ["GCP_PROJECT_ID"] = GCP_PROJECT_ID
    ee.Initialize(project=GCP_PROJECT_ID)
    print("Earth Engine ready — all sources available.")
else:
    print("Earth Engine disabled — sections 3-5 still work; section 6 will be skipped.")

## 3. Fetch daily climate data → pandas DataFrame

`nasa_power` uses plain HTTPS — no Earth Engine involved. We fetch one year of daily rainfall and temperature for Nairobi, Kenya. Swap in your own coordinates.

In [ ]:
from datetime import date

from climate_toolkit.fetch_data.source_data.sources.utils.models import ClimateVariable

LAT, LON = -1.286, 36.817  # Nairobi, Kenya — swap in your own site

VARS = [
    ClimateVariable.precipitation,
    ClimateVariable.max_temperature,
    ClimateVariable.min_temperature,
]

df = ct.fetch_climate_data(
    source="nasa_power",
    location_coord=(LAT, LON),
    variables=VARS,
    date_from=date(2020, 1, 1),
    date_to=date(2020, 12, 31),
    verbose=False,
)
df.head()

In [ ]:
# It's a normal DataFrame — plot, resample, export as usual.
df.set_index("date")["precipitation"].plot(
    figsize=(10, 3), title="Daily precipitation, Nairobi 2020 (NASA POWER)"
);

# ERA LTE workflow — climate vs yield, with visualizations

The ERA workflow scripts live in the repo's `examples/` folder (they ship with the
repo, **not** the pip wheel), so we clone the repo and run from it. The public
`lte_final.csv` is **auto-downloaded** by the scripts on first use.

In [ ]:
# Get the repo's `examples/` scripts. Works in Colab (clones the repo) and when
# run from the repo in VS Code / Jupyter (skips the clone — no nested copy).
import os

if os.path.exists("examples/era_lte_workflow.py"):
    pass                       # already at the repo root (e.g. VS Code)
elif os.path.exists("era_lte_workflow.py"):
    %cd ..                     # opened from examples/ -> go to repo root
else:
    !git clone -q https://github.com/CGIAR-Climate-Data-Hub/climate-toolkit.git
    %cd climate-toolkit        # fresh environment (e.g. Colab)

from IPython.display import Image        # for showing the plots inline

### Peek at `lte_final.csv`

A quick glance at the public ERA long-term-experiment table these workflows use. `era_fetch_data.py` downloads it once. Note the `Variety` (cultivar) column — it is often swapped over an LTE's life, so the section-4 trend plots mark where it changes.

In [ ]:
import pandas as pd
!python examples/era_fetch_data.py > /dev/null   # public ERA table, downloaded once
lte = pd.read_csv("lte_final.csv", low_memory=False)

# The crop and the cultivar/variety grown. Variety is often swapped over an LTE's
# life, so it is carried into the yield tables and marked on the section-4 plots.
lte[["Site.ID", "Product.Simple", "Variety"]].dropna(subset=["Variety"]).head()

## 1. List the ERA sites

In [ ]:
!python examples/era_lte_workflow.py examples/data/unique_sites_for_toolkit.csv --list-sites

## 2. Run the workflow — single site
(Use `--limit N` or `--site` to keep it fast; running *all* sites takes several minutes.)

In [ ]:
!python examples/era_lte_workflow.py examples/data/unique_sites_for_toolkit.csv --site "Awassa" --out single.csv > /dev/null
import pandas as pd
pd.read_csv("single.csv").head()

### Multiple sites

In [ ]:
!python examples/era_lte_workflow.py examples/data/unique_sites_for_toolkit.csv --site "Awassa" --site "Samaru" --out multi.csv > /dev/null
pd.read_csv("multi.csv").head()

## 3. Validation — toolkit rainfall vs ERA's own `rain_rain_sum`
Like-for-like over ERA's exact growing-season window. `lte_final.csv` auto-downloads.

In [ ]:
!python examples/era_final_validate.py lte_final.csv --source nasa_power --limit 60 --out era_final_compare.csv > /dev/null
!python examples/era_plot.py era_final_compare.csv --out era_validation.png > /dev/null
Image("era_validation.png")

### Which sites can I use for the trends?

Sections 4–5 draw on `lte_final.csv` — a *different, larger* set of sites (any with reported crop yield) than the ~100 shown by section 1's `--list-sites`. So look here, not in section 1, for the trend sites. The cell below prints them (and the exact count); pass any name to `--site` in the cells that follow.

**How `--site` matches:** case-insensitive **substring** — `--site "Makoka"` finds "Makoka ARS", so you don't need the full name. Keep it distinctive, though: a generic string like `"ARC"` matches several sites at once. When unsure, copy a name from the list below.

In [ ]:
import pandas as pd
from examples.era_fetch_data import maybe_fetch

_lf = pd.read_csv(maybe_fetch("lte_final.csv"), low_memory=False)
yield_sites = sorted(_lf.loc[_lf["Out.SubInd"] == "Crop Yield", "Site.ID"].dropna().astype(str).unique())
print(f"{len(yield_sites)} sites available for the yield trends — pass any (or a distinctive part) to --site:")
for s in yield_sites:
    print("  ", s)

## 4. Yield vs toolkit climate — trend figures
Toolkit variable as bars, one yield line per treatment, per site. We recommend trying various datasets, as some over- or underestimate rainfall. This will help identify which dataset correlates most with yields. Try `--source chirps_v2`, `chirps_v3_daily_rnl`, `agera_5`, or `era_5` instead of the default `nasa_power`. The cell below uses `era_5` (Earth Engine), so complete section 2 first — or change `--source` to `nasa_power` to run it credential-free. Dashed vertical guides on the plot mark where the crop **variety** changes over time.

In [ ]:
!python examples/era_yield_analysis.py lte_final.csv --source era_5 --site "Gourton" --out era_gourton.csv > /dev/null
!python examples/era_yield_plot.py era_gourton.csv --site "Gourton" --out era_gourton_trends.png > /dev/null
Image("era_gourton_trends.png")

### Match a specific practice subset (e.g. the NT nitrogen rates)

In [ ]:
!python examples/era_yield_plot.py era_gourton.csv --site "Gourton" --treatments "NT 0N,NT 100N,NT 200N" --out gourton_NT.png > /dev/null
Image("gourton_NT.png")

## 4b. Pick a specific crop for a multi-crop site
Some sites grew several crops (e.g. **Kouve** = cotton + maize). Use `--crop` to
choose one — otherwise you get whichever matched first.

In [ ]:
# Kouve, maize only (swap "Maize" for "cotton" for the other crop)
!python examples/era_yield_analysis.py lte_final.csv --site "Kouve" --crop "Maize" --out era_kouve_maize.csv > /dev/null
!python examples/era_yield_plot.py era_kouve_maize.csv --site "Kouve" --out era_kouve_maize_trends.png > /dev/null
Image("era_kouve_maize_trends.png")

## 4c. Uniquely-identified records (linked to master LTE ids)
The yield table leads with `lte_id, code, site_key, index` so every site-season
record is uniquely identified and linked to the master LTE registry
(`Code` → `LTE.ID`). (`Site.Lat.Unc`/`Site.Lon.Unc` aren't in `lte_final.csv`, so
they're not shown.)

In [ ]:
import pandas as pd
cols = ["lte_id","code","site_key","index","site_id","year","crop","treatment","yield_t_ha"]
pd.read_csv("era_kouve_maize.csv")[cols].head()

## 5. Combined multi-site view
One toolkit variable vs yield across several sites.

In [ ]:
!python examples/era_yield_analysis.py lte_final.csv --site "Zimuto"  --out era_Zimuto.csv > /dev/null
!python examples/era_yield_analysis.py lte_final.csv --site "Nyabeda" --out era_Nyabeda.csv > /dev/null
!python examples/era_yield_analysis.py lte_final.csv --site "Kouve"   --out era_Kouve.csv > /dev/null
!python examples/era_yield_multisite.py era_gourton.csv era_Zimuto.csv era_Nyabeda.csv era_Kouve.csv --variable tk_rain_total_mm --out era_multisite.png > /dev/null
Image("era_multisite.png")

---
### Note on data coverage
NASA POWER daily data starts ~**1981**, so ERA sites whose study years are entirely
in the 1970s are skipped (you'll see a `422` message for those — harmless). For
full historical coverage you can switch `--source` to an Earth Engine source such
as `agera_5` or `chirps_v3_daily_rnl` (you set Earth Engine up at the top).